In [1]:
from pathlib import Path
import duckdb

ORG_GLOB = (
    Path("../data/organizations")
    .resolve()
    .as_posix()
    + "/**/*.parquet"
)

print("Organization data path:")
print(ORG_GLOB)

Organization data path:
C:/Users/Gulso/Documents/Gul/CredBlock-FL/data/organizations/**/*.parquet


In [2]:
graph_sizes = duckdb.sql(f"""
    WITH entity_stats AS (
        SELECT
            organization,
            COUNT(*) AS login_records,
            COUNT(DISTINCT "User ID") AS unique_users,
            COUNT(DISTINCT "IP Address") AS unique_ips

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY organization
    ),

    pair_stats AS (
        SELECT
            organization,
            COUNT(*) AS unique_user_ip_pairs

        FROM (
            SELECT DISTINCT
                organization,
                "User ID",
                "IP Address"

            FROM read_parquet(
                '{ORG_GLOB}',
                hive_partitioning = true
            )
        )

        GROUP BY organization
    )

    SELECT
        e.organization,
        e.login_records,
        e.unique_users,
        e.unique_ips,

        e.unique_users + e.unique_ips
            AS potential_nodes,

        p.unique_user_ip_pairs,

        e.login_records - p.unique_user_ip_pairs
            AS repeated_interactions,

        ROUND(
            100.0 * p.unique_user_ip_pairs
            / e.login_records,
            2
        ) AS unique_pair_pct

    FROM entity_stats e

    JOIN pair_stats p
        USING (organization)

    ORDER BY organization
""").df()

display(graph_sizes)

,organization,login_records,unique_users,unique_ips,potential_nodes,unique_user_ip_pairs,repeated_interactions,unique_pair_pct
0,Org1,3080150,791567,733381,1524948,1697608,1382542,55.11
1,Org2,3250845,825185,762214,1587399,1787741,1463104,54.99
2,Org3,3520641,861765,796189,1657954,1889871,1630770,53.68
3,Org4,3611088,895714,821804,1717518,1975768,1635320,54.71
4,Org5,3780641,930625,852146,1782771,2068379,1712262,54.71


In [3]:
pair_frequency_stats = duckdb.sql(f"""
    WITH pair_counts AS (
        SELECT
            organization,
            "User ID",
            "IP Address",
            COUNT(*) AS interaction_count

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            "IP Address"
    )

    SELECT
        organization,

        COUNT(*) AS unique_user_ip_pairs,

        ROUND(
            AVG(interaction_count),
            2
        ) AS avg_interactions_per_pair,

        MIN(interaction_count) AS min_interactions,

        approx_quantile(
            interaction_count,
            0.50
        ) AS median_interactions,

        approx_quantile(
            interaction_count,
            0.90
        ) AS p90_interactions,

        approx_quantile(
            interaction_count,
            0.95
        ) AS p95_interactions,

        approx_quantile(
            interaction_count,
            0.99
        ) AS p99_interactions,

        MAX(interaction_count) AS max_interactions

    FROM pair_counts

    GROUP BY organization

    ORDER BY organization
""").df()

display(pair_frequency_stats)

,organization,unique_user_ip_pairs,avg_interactions_per_pair,min_interactions,median_interactions,p90_interactions,p95_interactions,p99_interactions,max_interactions
0,Org1,1697608,1.81,1,1,3,5,10,1204
1,Org2,1787741,1.82,1,1,3,5,10,1613
2,Org3,1889871,1.86,1,1,3,5,10,36413
3,Org4,1975768,1.83,1,1,3,5,10,1140
4,Org5,2068379,1.83,1,1,3,5,10,964


In [4]:
high_frequency_pairs = duckdb.sql(f"""
    WITH pair_counts AS (
        SELECT
            organization,
            "User ID",
            "IP Address",
            COUNT(*) AS interaction_count

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            "IP Address"
    )

    SELECT
        organization,

        COUNT(*) AS total_pairs,

        SUM(CASE
            WHEN interaction_count > 10
            THEN 1 ELSE 0
        END) AS pairs_over_10,

        SUM(CASE
            WHEN interaction_count > 100
            THEN 1 ELSE 0
        END) AS pairs_over_100,

        SUM(CASE
            WHEN interaction_count > 1000
            THEN 1 ELSE 0
        END) AS pairs_over_1000,

        ROUND(
            100.0 * SUM(CASE
                WHEN interaction_count > 10
                THEN 1 ELSE 0
            END) / COUNT(*),
            4
        ) AS pct_over_10,

        ROUND(
            100.0 * SUM(CASE
                WHEN interaction_count > 100
                THEN 1 ELSE 0
            END) / COUNT(*),
            4
        ) AS pct_over_100,

        ROUND(
            100.0 * SUM(CASE
                WHEN interaction_count > 1000
                THEN 1 ELSE 0
            END) / COUNT(*),
            6
        ) AS pct_over_1000

    FROM pair_counts

    GROUP BY organization

    ORDER BY organization
""").df()

display(high_frequency_pairs)

,organization,total_pairs,pairs_over_10,pairs_over_100,pairs_over_1000,pct_over_10,pct_over_100,pct_over_1000
0,Org1,1697608,15482.0,217.0,1.0,0.9120,0.0128,0.000059
1,Org2,1787741,16495.0,224.0,3.0,0.9227,0.0125,0.000168
2,Org3,1889871,17548.0,246.0,6.0,0.9285,0.0130,0.000317
3,Org4,1975768,18416.0,262.0,1.0,0.9321,0.0133,0.000051
4,Org5,2068379,19314.0,276.0,0.0,0.9338,0.0133,0.000000


In [5]:
top_repeated_pairs = duckdb.sql(f"""
    WITH pair_stats AS (
        SELECT
            organization,
            "User ID",
            "IP Address",

            COUNT(*) AS interaction_count,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            SUM(
                CASE WHEN "Is Attack IP" = TRUE
                THEN 1 ELSE 0 END
            ) AS attack_ip_logins,

            SUM(
                CASE WHEN "Is Account Takeover" = TRUE
                THEN 1 ELSE 0 END
            ) AS account_takeover_logins,

            MIN("Login Timestamp") AS first_login,

            MAX("Login Timestamp") AS last_login

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            "IP Address"
    ),

    ranked_pairs AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY organization
                ORDER BY interaction_count DESC
            ) AS rank_in_org

        FROM pair_stats
    )

    SELECT
        organization,
        rank_in_org,
        "User ID",
        "IP Address",
        interaction_count,
        successful_logins,
        failed_logins,
        attack_ip_logins,
        account_takeover_logins,
        first_login,
        last_login

    FROM ranked_pairs

    WHERE rank_in_org <= 5

    ORDER BY
        organization,
        rank_in_org
""").df()

display(top_repeated_pairs)

,organization,rank_in_org,User ID,IP Address,interaction_count,successful_logins,failed_logins,attack_ip_logins,account_takeover_logins,first_login,last_login
0,Org1,1,-8897190181838729192,84.208.244.73,1204,0.0,1204.0,0.0,0.0,2020-08-23 05:29:36.171,2021-02-28 21:30:27.255
1,Org1,2,4588325308801696666,10.0.77.220,797,771.0,26.0,0.0,0.0,2020-02-25 07:51:38.372,2021-02-26 07:19:01.599
2,Org1,3,3219851642254212624,10.0.77.169,753,751.0,2.0,0.0,0.0,2020-02-04 07:59:06.839,2021-02-24 07:29:15.379
3,Org1,4,-3031272205102765641,84.48.62.231,714,691.0,23.0,0.0,0.0,2020-11-14 12:09:28.166,2021-02-28 20:31:01.580
4,Org1,5,6665318783057062876,173.205.2.177,696,0.0,696.0,696.0,0.0,2020-06-04 16:05:20.828,2021-02-17 16:10:21.191
5,Org2,1,-2894367365566592989,95.169.53.178,1613,1550.0,63.0,0.0,0.0,2020-02-03 13:05:00.146,2021-02-26 06:22:30.720
6,Org2,2,7306141804897825369,84.209.121.191,1121,1102.0,19.0,0.0,0.0,2020-02-03 14:30:18.578,2021-02-04 10:21:37.372
7,Org2,3,6847222853559127056,10.3.232.33,1012,1005.0,7.0,0.0,0.0,2020-02-04 08:20:53.263,2020-10-14 11:06:33.047
8,Org2,4,7941119571361817989,141.195.15.229,839,804.0,35.0,0.0,0.0,2020-02-04 08:43:39.895,2021-02-15 15:21:37.616
9,Org2,5,-4652302824165996131,62.197.64.29,753,732.0,21.0,0.0,0.0,2020-06-09 07:34:34.052,2021-02-26 14:36:06.415


In [6]:
hourly_pair_stats = duckdb.sql(f"""
    WITH hourly_counts AS (
        SELECT
            organization,
            "User ID",
            "IP Address",
            DATE_TRUNC(
                'hour',
                "Login Timestamp"
            ) AS hour_window,

            COUNT(*) AS interactions_in_hour,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_in_hour,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_in_hour

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            "IP Address",
            hour_window
    )

    SELECT
        organization,

        COUNT(*) AS pair_hour_windows,

        ROUND(
            AVG(interactions_in_hour),
            2
        ) AS avg_interactions_per_hour,

        approx_quantile(
            interactions_in_hour,
            0.50
        ) AS median_interactions,

        approx_quantile(
            interactions_in_hour,
            0.90
        ) AS p90_interactions,

        approx_quantile(
            interactions_in_hour,
            0.95
        ) AS p95_interactions,

        approx_quantile(
            interactions_in_hour,
            0.99
        ) AS p99_interactions,

        MAX(interactions_in_hour)
            AS max_interactions_in_hour

    FROM hourly_counts

    GROUP BY organization

    ORDER BY organization
""").df()

display(hourly_pair_stats)

,organization,pair_hour_windows,avg_interactions_per_hour,median_interactions,p90_interactions,p95_interactions,p99_interactions,max_interactions_in_hour
0,Org1,2452497,1.26,1,2,3,5,38
1,Org2,2584306,1.26,1,2,3,5,287
2,Org3,2741718,1.28,1,2,3,5,71
3,Org4,2865273,1.26,1,2,3,5,50
4,Org5,2996349,1.26,1,2,3,5,61


In [7]:
ten_minute_pair_stats = duckdb.sql(f"""
    WITH ten_minute_counts AS (
        SELECT
            organization,
            "User ID",
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS interactions_in_window,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_in_window,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_in_window

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            "IP Address",
            ten_minute_window
    )

    SELECT
        organization,

        COUNT(*) AS pair_10min_windows,

        ROUND(
            AVG(interactions_in_window),
            2
        ) AS avg_interactions,

        approx_quantile(
            interactions_in_window,
            0.50
        ) AS median_interactions,

        approx_quantile(
            interactions_in_window,
            0.90
        ) AS p90_interactions,

        approx_quantile(
            interactions_in_window,
            0.95
        ) AS p95_interactions,

        approx_quantile(
            interactions_in_window,
            0.99
        ) AS p99_interactions,

        MAX(interactions_in_window)
            AS max_interactions_in_10min

    FROM ten_minute_counts

    GROUP BY organization

    ORDER BY organization
""").df()

display(ten_minute_pair_stats)

,organization,pair_10min_windows,avg_interactions,median_interactions,p90_interactions,p95_interactions,p99_interactions,max_interactions_in_10min
0,Org1,2643571,1.17,1,2,2,4,35
1,Org2,2787082,1.17,1,2,2,4,56
2,Org3,2996056,1.18,1,2,2,4,31
3,Org4,3092054,1.17,1,2,2,4,35
4,Org5,3235482,1.17,1,2,2,4,37


In [8]:
ip_user_window_stats = duckdb.sql(f"""
    WITH ip_windows AS (
        SELECT
            organization,
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS login_attempts,

            COUNT(DISTINCT "User ID")
                AS distinct_users,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "IP Address",
            ten_minute_window
    )

    SELECT
        organization,

        COUNT(*) AS ip_time_windows,

        ROUND(
            AVG(distinct_users),
            2
        ) AS avg_users_per_ip,

        approx_quantile(
            distinct_users,
            0.50
        ) AS median_users,

        approx_quantile(
            distinct_users,
            0.90
        ) AS p90_users,

        approx_quantile(
            distinct_users,
            0.95
        ) AS p95_users,

        approx_quantile(
            distinct_users,
            0.99
        ) AS p99_users,

        MAX(distinct_users)
            AS max_users_per_ip_10min

    FROM ip_windows

    GROUP BY organization

    ORDER BY organization
""").df()

display(ip_user_window_stats)

,organization,ip_time_windows,avg_users_per_ip,median_users,p90_users,p95_users,p99_users,max_users_per_ip_10min
0,Org1,2630847,1.00,1,1,1,1,17
1,Org2,2772834,1.01,1,1,1,1,25
2,Org3,2979687,1.01,1,1,1,1,15
3,Org4,3073303,1.01,1,1,1,1,22
4,Org5,3216733,1.01,1,1,1,1,19


In [9]:
multi_user_ip_windows = duckdb.sql(f"""
    WITH ip_windows AS (
        SELECT
            organization,
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS login_attempts,

            COUNT(DISTINCT "User ID")
                AS distinct_users,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "IP Address",
            ten_minute_window
    )

    SELECT
        organization,

        COUNT(*) AS total_ip_windows,

        SUM(
            CASE WHEN distinct_users > 1
            THEN 1 ELSE 0 END
        ) AS windows_over_1_user,

        SUM(
            CASE WHEN distinct_users > 5
            THEN 1 ELSE 0 END
        ) AS windows_over_5_users,

        SUM(
            CASE WHEN distinct_users > 10
            THEN 1 ELSE 0 END
        ) AS windows_over_10_users,

        ROUND(
            100.0 * SUM(
                CASE WHEN distinct_users > 1
                THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        ) AS pct_over_1_user,

        ROUND(
            100.0 * SUM(
                CASE WHEN distinct_users > 5
                THEN 1 ELSE 0 END
            ) / COUNT(*),
            6
        ) AS pct_over_5_users,

        ROUND(
            100.0 * SUM(
                CASE WHEN distinct_users > 10
                THEN 1 ELSE 0 END
            ) / COUNT(*),
            6
        ) AS pct_over_10_users

    FROM ip_windows

    GROUP BY organization

    ORDER BY organization
""").df()

display(multi_user_ip_windows)

,organization,total_ip_windows,windows_over_1_user,windows_over_5_users,windows_over_10_users,pct_over_1_user,pct_over_5_users,pct_over_10_users
0,Org1,2630847,11744.0,74.0,10.0,0.4464,0.002813,0.000380
1,Org2,2772834,13223.0,77.0,12.0,0.4769,0.002777,0.000433
2,Org3,2979687,14996.0,94.0,22.0,0.5033,0.003155,0.000738
3,Org4,3073303,17063.0,124.0,22.0,0.5552,0.004035,0.000716
4,Org5,3216733,17529.0,83.0,23.0,0.5449,0.002580,0.000715


In [10]:
top_multi_user_windows = duckdb.sql(f"""
    WITH ip_windows AS (
        SELECT
            organization,
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS login_attempts,

            COUNT(DISTINCT "User ID")
                AS distinct_users,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Is Attack IP" = TRUE
                THEN 1 ELSE 0 END
            ) AS attack_ip_logins,

            SUM(
                CASE WHEN "Is Account Takeover" = TRUE
                THEN 1 ELSE 0 END
            ) AS account_takeover_logins

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "IP Address",
            ten_minute_window
    ),

    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY organization
                ORDER BY
                    distinct_users DESC,
                    login_attempts DESC
            ) AS rank_in_org

        FROM ip_windows
    )

    SELECT
        organization,
        rank_in_org,
        "IP Address",
        ten_minute_window,
        login_attempts,
        distinct_users,
        failed_logins,
        successful_logins,
        attack_ip_logins,
        account_takeover_logins

    FROM ranked

    WHERE rank_in_org <= 5

    ORDER BY
        organization,
        rank_in_org
""").df()

display(top_multi_user_windows)

,organization,rank_in_org,IP Address,ten_minute_window,login_attempts,distinct_users,failed_logins,successful_logins,attack_ip_logins,account_takeover_logins
0,Org1,1,10.0.160.77,2020-10-13 07:50:00,18,17,5.0,13.0,0.0,0.0
1,Org1,2,10.0.160.77,2020-08-12 11:30:00,17,17,0.0,17.0,0.0,0.0
2,Org1,3,10.0.160.77,2020-08-12 11:10:00,15,13,4.0,11.0,0.0,0.0
3,Org1,4,10.0.160.77,2020-08-12 11:50:00,12,12,1.0,11.0,0.0,0.0
4,Org1,5,10.0.160.77,2021-02-11 07:10:00,12,12,3.0,9.0,0.0,0.0
5,Org2,1,10.0.160.77,2020-10-13 07:50:00,25,25,3.0,22.0,0.0,0.0
6,Org2,2,10.0.160.77,2020-10-13 07:40:00,18,17,4.0,14.0,0.0,0.0
7,Org2,3,10.0.160.77,2020-08-12 11:30:00,16,16,3.0,13.0,0.0,0.0
8,Org2,4,10.0.160.77,2020-08-12 11:40:00,16,16,4.0,12.0,0.0,0.0
9,Org2,5,10.0.160.77,2020-08-12 11:00:00,16,16,4.0,12.0,0.0,0.0


In [11]:
user_ip_window_stats = duckdb.sql(f"""
    WITH user_windows AS (
        SELECT
            organization,
            "User ID",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS login_attempts,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            ten_minute_window
    )

    SELECT
        organization,

        COUNT(*) AS user_time_windows,

        ROUND(
            AVG(distinct_ips),
            2
        ) AS avg_ips_per_user,

        approx_quantile(
            distinct_ips,
            0.50
        ) AS median_ips,

        approx_quantile(
            distinct_ips,
            0.90
        ) AS p90_ips,

        approx_quantile(
            distinct_ips,
            0.95
        ) AS p95_ips,

        approx_quantile(
            distinct_ips,
            0.99
        ) AS p99_ips,

        MAX(distinct_ips)
            AS max_ips_per_user_10min

    FROM user_windows

    GROUP BY organization

    ORDER BY organization
""").df()

display(user_ip_window_stats)

,organization,user_time_windows,avg_ips_per_user,median_ips,p90_ips,p95_ips,p99_ips,max_ips_per_user_10min
0,Org1,2616634,1.01,1,1,1,2,8
1,Org2,2758658,1.01,1,1,1,1,7
2,Org3,2965202,1.01,1,1,1,2,7
3,Org4,3059743,1.01,1,1,1,2,8
4,Org5,3201117,1.01,1,1,1,2,8


In [12]:
user_ip_window_stats = duckdb.sql(f"""
    WITH user_windows AS (
        SELECT
            organization,
            "User ID",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS login_attempts,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            ten_minute_window
    )

    SELECT
        organization,

        COUNT(*) AS user_time_windows,

        ROUND(
            AVG(distinct_ips),
            2
        ) AS avg_ips_per_user,

        approx_quantile(
            distinct_ips,
            0.50
        ) AS median_ips,

        approx_quantile(
            distinct_ips,
            0.90
        ) AS p90_ips,

        approx_quantile(
            distinct_ips,
            0.95
        ) AS p95_ips,

        approx_quantile(
            distinct_ips,
            0.99
        ) AS p99_ips,

        MAX(distinct_ips)
            AS max_ips_per_user_10min

    FROM user_windows

    GROUP BY organization

    ORDER BY organization
""").df()

display(user_ip_window_stats)

,organization,user_time_windows,avg_ips_per_user,median_ips,p90_ips,p95_ips,p99_ips,max_ips_per_user_10min
0,Org1,2616634,1.01,1,1,1,1,8
1,Org2,2758658,1.01,1,1,1,2,7
2,Org3,2965202,1.01,1,1,1,2,7
3,Org4,3059743,1.01,1,1,1,2,8
4,Org5,3201117,1.01,1,1,1,2,8


In [13]:
graph_sizes = duckdb.sql(f"""
    WITH entity_stats AS (
        SELECT
            organization,
            COUNT(*) AS login_records,
            COUNT(DISTINCT "User ID") AS unique_users,
            COUNT(DISTINCT "IP Address") AS unique_ips

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY organization
    ),

    pair_stats AS (
        SELECT
            organization,
            COUNT(*) AS unique_user_ip_pairs

        FROM (
            SELECT DISTINCT
                organization,
                "User ID",
                "IP Address"

            FROM read_parquet(
                '{ORG_GLOB}',
                hive_partitioning = true
            )
        )

        GROUP BY organization
    )

    SELECT
        e.organization,
        e.login_records,
        e.unique_users,
        e.unique_ips,

        e.unique_users + e.unique_ips
            AS potential_nodes,

        p.unique_user_ip_pairs,

        e.login_records - p.unique_user_ip_pairs
            AS repeated_interactions,

        ROUND(
            100.0 * p.unique_user_ip_pairs
            / e.login_records,
            2
        ) AS unique_pair_pct

    FROM entity_stats e

    JOIN pair_stats p
        USING (organization)

    ORDER BY organization
""").df()

display(graph_sizes)

,organization,login_records,unique_users,unique_ips,potential_nodes,unique_user_ip_pairs,repeated_interactions,unique_pair_pct
0,Org1,3080150,791567,733381,1524948,1697608,1382542,55.11
1,Org2,3250845,825185,762214,1587399,1787741,1463104,54.99
2,Org3,3520641,861765,796189,1657954,1889871,1630770,53.68
3,Org4,3611088,895714,821804,1717518,1975768,1635320,54.71
4,Org5,3780641,930625,852146,1782771,2068379,1712262,54.71


In [14]:
top_multi_ip_user_windows = duckdb.sql(f"""
    WITH user_windows AS (
        SELECT
            organization,
            "User ID",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS login_attempts,

            COUNT(DISTINCT "IP Address")
                AS distinct_ips,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_logins,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_logins,

            SUM(
                CASE WHEN "Is Attack IP" = TRUE
                THEN 1 ELSE 0 END
            ) AS attack_ip_logins,

            SUM(
                CASE WHEN "Is Account Takeover" = TRUE
                THEN 1 ELSE 0 END
            ) AS account_takeover_logins

        FROM read_parquet(
            '{ORG_GLOB}',
            hive_partitioning = true
        )

        GROUP BY
            organization,
            "User ID",
            ten_minute_window
    ),

    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY organization
                ORDER BY
                    distinct_ips DESC,
                    login_attempts DESC
            ) AS rank_in_org

        FROM user_windows
    )

    SELECT
        organization,
        rank_in_org,
        "User ID",
        ten_minute_window,
        login_attempts,
        distinct_ips,
        failed_logins,
        successful_logins,
        attack_ip_logins,
        account_takeover_logins

    FROM ranked

    WHERE rank_in_org <= 5

    ORDER BY
        organization,
        rank_in_org
""").df()

display(top_multi_ip_user_windows)

,organization,rank_in_org,User ID,ten_minute_window,login_attempts,distinct_ips,failed_logins,successful_logins,attack_ip_logins,account_takeover_logins
0,Org1,1,-6262466877883415483,2020-10-15 04:30:00,12,8,8.0,4.0,8.0,0.0
1,Org1,2,-6562934789786736189,2020-11-30 16:00:00,14,7,13.0,1.0,12.0,0.0
2,Org1,3,4598050714095864788,2021-01-24 17:40:00,10,7,8.0,2.0,8.0,0.0
3,Org1,4,-6562934789786736189,2020-12-01 08:40:00,14,6,13.0,1.0,12.0,0.0
4,Org1,5,5377601694720060292,2020-12-05 23:50:00,13,6,11.0,2.0,11.0,0.0
5,Org2,1,-4087676184606077266,2021-02-09 15:50:00,8,7,8.0,0.0,8.0,0.0
6,Org2,2,-3439684067780607665,2020-04-06 05:30:00,7,7,3.0,4.0,0.0,0.0
7,Org2,3,-2670881433238306703,2020-11-24 20:50:00,10,6,10.0,0.0,9.0,0.0
8,Org2,4,-2868985086423378997,2021-01-28 20:20:00,9,6,9.0,0.0,7.0,0.0
9,Org2,5,-1402828506185574531,2020-07-19 17:00:00,8,6,8.0,0.0,7.0,0.0
